# minissp on Kaggle — Search Self-Play (arXiv:2510.18821), end to end

This notebook builds the data and retrieval index, smoke-tests the rollout, measures the
base model, trains with search self-play, and produces the results: curves, eval accuracy
with 95% CIs, and the README's success-bar checks.

**Notebook settings (right sidebar → Session options):**
- **Accelerator: GPU T4 x2.** The policy goes on GPU 0 and the 4-bit judge on GPU 1.
  A single T4 also works. P100 is not supported (bitsandbytes 4-bit).
- **Internet: On.** The notebook clones the repo and downloads models and data from Hugging Face.

**How to run:** use **Save Version → Save & Run All (Commit)**. That runs in the
background for up to 12 hours, and everything under `/kaggle/working` becomes the version's output.

**Time:** about 1–1.5 h for one-time data prep (downloading and scanning wiki-18, ~13 GB),
then training. Each 10-step chunk ends with an eval on 300 questions. The notebook stops
cleanly before Kaggle's 12-hour limit (`TIME_BUDGET_HOURS`).

**If it didn't reach `MAX_STEPS` in one session:** open the notebook, choose **Add Input →
Your Work → this notebook's latest version**, and commit again. The next run finds the
previous data, index and checkpoints under `/kaggle/input`, skips prep, and resumes training.

In [ ]:
# ---- Configuration -------------------------------------------------------------
REPO_URL = "https://github.com/pranathitaddi/minissp.git"
BRANCH = "kaggle-notebook"   # branch with the Kaggle fixes; switch to "main" once merged

RUN_ID = "kaggle-001"      # change for a new, independent run (e.g. an ablation)
MAX_STEPS = 50             # README success bar is 50 steps
EVAL_EVERY = 10            # eval on 300 held-out questions every N steps
TIME_BUDGET_HOURS = 11.0   # Kaggle kills sessions at 12 h; leave room to write outputs

ANSWER_SAMPLE = 6000       # train answers to scope the corpus around (freeze draws 800)
SMOKE_MIN_FORMAT_VALID = 0.2   # abort before training if the base model can't follow the tag format

# Extra flags forwarded to train/eval, e.g. the paper's Table 3 ablation:
#   EXTRA_ARGS = ["--noise-docs", "0"]     (use a different RUN_ID per setting)
EXTRA_ARGS = []

In [ ]:
# ---- Helpers -------------------------------------------------------------------
import glob
import json
import os
import shlex
import shutil
import subprocess
import threading
import time

T0 = time.time()
WORK = "/kaggle/working"
REPO = f"{WORK}/minissp"
DATA = f"{REPO}/data"
INDEX = f"{WORK}/index"
RUNS = f"{WORK}/runs"
RUN_DIR = f"{RUNS}/{RUN_ID}"
SCRATCH = "/tmp/minissp_scratch"          # big temporary files; NOT saved as output
RESULTS = f"{WORK}/results"

# Model weights (~10 GB) live outside /kaggle/working, which is capped at 20 GB.
os.environ["HF_HOME"] = "/tmp/hf_home"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONUNBUFFERED"] = "1"
os.makedirs(SCRATCH, exist_ok=True)


def elapsed_h():
    return (time.time() - T0) / 3600


def sh(cmd, timeout=None, cwd=REPO):
    # Runs a shell command, streams its output live, returns it, raises on failure.
    print(f"$ {cmd}", flush=True)
    proc = subprocess.Popen(cmd, shell=True, cwd=cwd, text=True, bufsize=1,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    timer = None
    if timeout is not None:
        timer = threading.Timer(max(1, timeout), proc.kill)
        timer.start()
    lines = []
    for line in proc.stdout:
        print(line, end="", flush=True)
        lines.append(line)
    proc.wait()
    if timer is not None:
        timed_out = not timer.is_alive()
        timer.cancel()
        if timed_out:
            raise TimeoutError(f"killed after {timeout:.0f}s: {cmd}")
    if proc.returncode != 0:
        raise RuntimeError(f"exit code {proc.returncode}: {cmd}")
    return "".join(lines)


def latest_step():
    path = f"{RUN_DIR}/latest"
    if not os.path.exists(path):
        return 0
    text = open(path).read().strip()
    return int(text) if text else 0

## 1. Code and dependencies

In [ ]:
if os.path.isdir(f"{REPO}/.git"):
    sh(f"git fetch -q origin {BRANCH} && git checkout -q {BRANCH} && git reset -q --hard origin/{BRANCH}")
else:
    sh(f"git clone -q --branch {BRANCH} {REPO_URL} {REPO}", cwd=WORK)
sh("git log -1 --oneline")

# Fail fast if the pushed code predates the fixes this notebook relies on.
_src = {f: open(f"{REPO}/minissp/{f}").read() for f in ("train.py", "config.py", "data.py")}
assert "replace(item, proposer_idx=None)" in _src["train.py"], \
    "Cloned repo lacks the replay-buffer reward fix. Push your local changes, then re-run."
assert "judge_device" in _src["config.py"], "Cloned repo lacks --judge-device. Push your local changes."
assert "_build_text_matcher" in _src["data.py"], "Cloned repo lacks the fast corpus scoper. Push your local changes."

sh("pip install -q -r requirements.txt && pip install -q -e . --no-deps")

In [ ]:
import torch

n_gpu = torch.cuda.device_count()
assert n_gpu >= 1, "No GPU. Set Accelerator to 'GPU T4 x2' in the notebook settings."
names = [torch.cuda.get_device_name(i) for i in range(n_gpu)]
print("GPUs:", names)
assert not any("P100" in n for n in names), "P100 can't run the 4-bit judge. Switch the accelerator to GPU T4 x2."

JUDGE_DEVICE = "cuda:1" if n_gpu >= 2 else ""
COMMON = ["--run-id", RUN_ID, "--index-dir", INDEX, "--runs-root", RUNS] + EXTRA_ARGS
if JUDGE_DEVICE:
    COMMON += ["--judge-device", JUDGE_DEVICE]
COMMON_STR = " ".join(shlex.quote(a) for a in COMMON)
print("judge device:", JUDGE_DEVICE or "same GPU as policy")

## 2. Pick up earlier work (if this notebook's previous output is attached as an input)

Restores the data files, the index and this run's checkpoints, so a continuation skips prep
and resumes training where the last session stopped.

In [ ]:
def find_input(rel):
    hits = sorted(glob.glob(f"/kaggle/input/**/{rel}", recursive=True))
    return hits[0] if hits else None

os.makedirs(DATA, exist_ok=True)
for name in ["train.jsonl", "test.jsonl", "corpus_scoped.jsonl", "corpus_ids.json",
             "answers_toy.jsonl", "eval_toy.jsonl", "manifest.json"]:
    dst = f"{DATA}/{name}"
    src = find_input(f"minissp/data/{name}")
    if src and not os.path.exists(dst):
        shutil.copy(src, dst)
        print("restored", name)

src = find_input("index/corpus.faiss")
if src and not os.path.exists(f"{INDEX}/corpus.faiss"):
    shutil.copytree(os.path.dirname(src), INDEX, dirs_exist_ok=True)
    print("restored index")

src = find_input(f"runs/{RUN_ID}/latest")
if src and not os.path.exists(f"{RUN_DIR}/latest"):
    shutil.copytree(os.path.dirname(src), RUN_DIR, dirs_exist_ok=True)
    print(f"restored run {RUN_ID} at step {latest_step()}")

print("starting from step", latest_step())

## 3. Data and retrieval index (one time, ~1–1.5 h)

1. Answer set D and test set from `Quark-LLM/SSP` (paper §A.4).
2. wiki-18 (21M passages, the paper's corpus), scanned once and scoped to passages that
   mention the sampled answers or the NQ/HotpotQA eval targets, plus 20k random distractors.
3. `freeze`: 800 training answers (coverage ≥ 3) and 300 eval questions (150 NQ + 150 HotpotQA).
4. e5-base-v2 embeddings → FAISS index.

In [ ]:
def have(*names):
    return all(os.path.exists(f"{DATA}/{n}") for n in names)

if not have("train.jsonl", "test.jsonl"):
    sh("python -m minissp.data download --skip-wiki")

if not have("corpus_ids.json", "corpus_scoped.jsonl"):
    free_gb = shutil.disk_usage("/tmp").free / 1e9
    print(f"/tmp free: {free_gb:.0f} GB")
    from huggingface_hub import hf_hub_download
    wiki = hf_hub_download("PeterJinGo/wiki-18-corpus", "wiki-18.jsonl.gz",
                           repo_type="dataset", local_dir=f"{SCRATCH}/wiki")
    sh(f"python -m minissp.data scope --wiki {wiki} --answer-sample {ANSWER_SAMPLE} "
       f"--eval-sources nq,hotpotqa")
    shutil.rmtree(f"{SCRATCH}/wiki", ignore_errors=True)
    shutil.rmtree(os.environ["HF_HOME"] + "/hub/datasets--PeterJinGo--wiki-18-corpus", ignore_errors=True)

if not have("answers_toy.jsonl", "eval_toy.jsonl", "manifest.json"):
    sh("python -m minissp.data freeze")

if not os.path.exists(f"{INDEX}/corpus.faiss"):
    sh(f"python -m minissp.data build-index --out {INDEX}")

manifest = json.load(open(f"{DATA}/manifest.json"))
print(json.dumps(manifest, indent=2))
assert manifest["eval_toy_count"] >= 200, "Too few eval questions; raise ANSWER_SAMPLE or check scoping."
assert manifest["answers_toy_count"] >= 400, "Too few training answers; raise ANSWER_SAMPLE."
n_docs = sum(1 for _ in open(f"{INDEX}/docs.jsonl"))
print(f"index: {n_docs:,} passages")

## 4. Smoke test: does the base model follow the tag protocol?

Loads policy + judge + retriever together and runs 8 solver and 8 proposer rollouts.
A `format_valid_rate` near 0 means a prompt or chat-template bug, which training won't fix.

In [ ]:
import re

if latest_step() == 0:
    judge_flag = f"--judge-device {JUDGE_DEVICE}" if JUDGE_DEVICE else ""
    out = sh(f"python -m minissp.rollout --smoke --n 8 --index-dir {INDEX} "
             f"--out {RUNS}/smoke {judge_flag}")
    fvr = float(re.search(r"format_valid_rate\s+([0-9.]+)", out).group(1))
    if fvr < SMOKE_MIN_FORMAT_VALID:
        raise SystemExit(f"format_valid_rate {fvr:.2f} < {SMOKE_MIN_FORMAT_VALID}: "
                         f"inspect {RUNS}/smoke/samples/smoke.jsonl before spending GPU time.")
    print("\nTwo proposer samples:")
    rows = [json.loads(line) for line in open(f"{RUNS}/smoke/samples/smoke.jsonl")]
    for r in [r for r in rows if r["role"] == "proposer"][:2]:
        print("-" * 80, "\n", r["text"][-1500:])
else:
    print("resuming a run; smoke test skipped")

## 5. Baseline: evaluate the untrained policy (step 0)

In [ ]:
if not os.path.exists(f"{RUN_DIR}/eval_step_0.json"):
    sh(f"python -m minissp.eval {COMMON_STR} --step 0")
print(json.dumps(json.load(open(f"{RUN_DIR}/eval_step_0.json")), indent=2))

## 6. Search self-play training

Runs in chunks that end on an eval step (10, 20, …). Before each chunk the notebook checks
there's time for another one, so it never gets killed mid-chunk. Every step is
checkpointed, and a re-run resumes from `runs/<RUN_ID>/latest`.

In [ ]:
budget_s = TIME_BUDGET_HOURS * 3600
last_chunk_s = None
stop_reason = None

while (start := latest_step()) < MAX_STEPS:
    target = min(MAX_STEPS, (start // EVAL_EVERY + 1) * EVAL_EVERY)
    remaining = budget_s - (time.time() - T0)
    if last_chunk_s is not None and remaining < 1.15 * last_chunk_s:
        stop_reason = f"time budget: {remaining/60:.0f} min left, last chunk took {last_chunk_s/60:.0f} min"
        break
    if remaining < 20 * 60:
        stop_reason = "time budget: under 20 min left"
        break
    print(f"\n=== steps {start + 1}..{target}  (elapsed {elapsed_h():.2f} h) ===")
    t = time.time()
    try:
        sh(f"python -m minissp.train {COMMON_STR} --max-steps {target} "
           f"--eval-every {EVAL_EVERY}", timeout=remaining - 10 * 60)
    except TimeoutError as e:
        stop_reason = f"hard timeout ({e}); resumes from the last checkpoint next session"
        break
    last_chunk_s = time.time() - t
    if latest_step() == start:
        stop_reason = "no progress (repeated skipped updates); inspect metrics.jsonl"
        break

print(f"\nstopped at step {latest_step()}/{MAX_STEPS}  ({elapsed_h():.2f} h)",
      f"- {stop_reason}" if stop_reason else "")
if latest_step() < MAX_STEPS:
    print("To continue: Add Input -> this notebook's output, then Save & Run All again.")

In [ ]:
# Final eval if the last step wasn't an eval step (e.g. MAX_STEPS not a multiple of EVAL_EVERY).
last = latest_step()
if last > 0 and not os.path.exists(f"{RUN_DIR}/eval_step_{last}.json") \
        and budget_s - (time.time() - T0) > 40 * 60:
    sh(f"python -m minissp.eval {COMMON_STR} --step {last}")

## 7. Results

Absolute accuracy isn't comparable to the paper's Table 1: the model is smaller, the corpus
is scoped, and the eval set is 300 questions. What carries over are the **dynamics** (paper
Fig. 3): solver reward rises then dips as the proposer learns to write harder questions,
proposer reward climbs, the RAG-verified rate holds or rises, and held-out accuracy doesn't fall.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

os.makedirs(RESULTS, exist_ok=True)
m = pd.read_json(f"{RUN_DIR}/metrics.jsonl", lines=True) if os.path.exists(f"{RUN_DIR}/metrics.jsonl") else pd.DataFrame()

evals = []
for p in glob.glob(f"{RUN_DIR}/eval_step_*.json"):
    e = json.load(open(p))
    evals.append({"step": e["step"], "acc": e["acc"], "ci_lo": e["ci95"][0], "ci_hi": e["ci95"][1],
                  **{f"acc_{k}": v["acc"] for k, v in e["by_source"].items()},
                  "format_valid_rate": e["format_valid_rate"]})
ev = pd.DataFrame(evals).sort_values("step").reset_index(drop=True) if evals else pd.DataFrame()

print("Held-out eval (300 questions: 150 NQ + 150 HotpotQA), greedy, judge-scored:")
display(ev.round(3)) if len(ev) else print("no eval files yet")

In [ ]:
BLUE, ORANGE = "#2a78d6", "#eb6834"      # categorical slots 1 and 2, fixed order
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e6e5e0"

def smooth(s, w=5):
    return s.rolling(w, min_periods=1).mean()

def style(ax, title, ylim=None):
    ax.set_title(title, loc="left", fontsize=11, color=INK)
    ax.grid(axis="y", color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(GRID)
    ax.tick_params(colors=MUTED, labelsize=9)
    ax.set_xlabel("step", color=MUTED, fontsize=9)
    if ylim:
        ax.set_ylim(*ylim)

def line(ax, x, y, color, label):
    ax.plot(x, y, color=color, alpha=0.25, linewidth=1)          # raw
    ax.plot(x, smooth(y), color=color, linewidth=2, label=label)  # 5-step mean

panels = [
    ("In-game reward (cf. paper Fig. 3a)", [("solver_acc", "solver reward"), ("proposer_mean_reward", "proposer reward")]),
    ("Question filtering", [("rag_verified_rate", "RAG-verified"), ("rule_pass_rate", "rule-filter pass")]),
    ("Format validity", [("format_valid_rate", "format valid")]),
    ("GRPO groups with signal", [("frac_groups_nonzero_var", "groups w/ reward variance")]),
    ("Search calls per trajectory", [("avg_turns_solver", "solver"), ("avg_turns_proposer", "proposer")]),
]
fig, axes = plt.subplots(2, 3, figsize=(15, 8), facecolor="#fcfcfb")
axes = axes.ravel()
if len(m):
    for ax, (title, series) in zip(axes, panels):
        for (col, label), color in zip(series, (BLUE, ORANGE)):
            line(ax, m["step"], m[col], color, label)
        style(ax, title, (0, 3.2) if "Search" in title else (-0.02, 1.02))  # 3 = max search turns
        if len(series) > 1:
            ax.legend(frameon=False, fontsize=9, labelcolor=MUTED)

ax = axes[5]
if len(ev):
    yerr = [ev["acc"] - ev["ci_lo"], ev["ci_hi"] - ev["acc"]]
    ax.errorbar(ev["step"], ev["acc"], yerr=yerr, color=BLUE, linewidth=2, marker="o",
                markersize=7, capsize=4, elinewidth=1.2)
    for _, r in ev.iterrows():
        ax.annotate(f"{r['acc']:.3f}", (r["step"], r["ci_hi"]), textcoords="offset points",
                    xytext=(0, 4), ha="center", fontsize=8, color=MUTED)
style(ax, "Held-out accuracy, 95% bootstrap CI")
for a in axes:
    a.set_facecolor("#fcfcfb")
fig.tight_layout()
fig.savefig(f"{RESULTS}/training_curves.png", dpi=150)
plt.show()

### Success bar (from the README)

- 50 contiguous steps, no duplicated or missing step in `metrics.jsonl`
- no collapse (proposer reward → 1 with solver accuracy → 0 for 3+ steps)
- `rag_verified_rate` non-decreasing in trend
- final eval accuracy not below step 0's 95% CI

In [ ]:
import numpy as np

checks = {}
if len(m):
    steps = m["step"].tolist()
    checks["reached MAX_STEPS"] = latest_step() >= MAX_STEPS
    checks["steps contiguous, no duplicates"] = steps == list(range(1, len(steps) + 1))
    collapsed = ((m["proposer_mean_reward"] >= 0.95) & (m["solver_acc"] <= 0.05)).astype(int)
    longest = collapsed.groupby((collapsed != collapsed.shift()).cumsum()).sum().max()
    checks["no collapse (3+ steps)"] = bool(longest < 3)
    slope = np.polyfit(m["step"], m["rag_verified_rate"], 1)[0] if len(m) > 1 else 0.0
    checks[f"rag_verified_rate trend >= 0 (slope {slope:+.4f}/step)"] = bool(slope >= 0)
if len(ev) and (ev["step"] == 0).any() and ev["step"].max() > 0:
    base = ev[ev["step"] == 0].iloc[0]
    final = ev.iloc[-1]
    checks[f"final eval {final['acc']:.3f} (step {int(final['step'])}) >= step-0 CI low {base['ci_lo']:.3f}"] = \
        bool(final["acc"] >= base["ci_lo"])
    checks[f"final eval above step-0 CI high {base['ci_hi']:.3f} (a real improvement)"] = \
        bool(final["acc"] > base["ci_hi"])

for name, ok in checks.items():
    print(f"[{'PASS' if ok else 'FAIL'}] {name}")

In [ ]:
# A few questions the trained proposer wrote (fresh, RAG-verified) in the last steps.
sample_files = sorted(glob.glob(f"{RUN_DIR}/samples/step_*.jsonl"))[-3:]
shown = 0
for p in reversed(sample_files):
    for r in map(json.loads, open(p)):
        if not r["is_dummy"] and shown < 8:
            print(f"Q: {r['question']}\n   answer: {r['answer']}\n")
            shown += 1

summary = {
    "run_id": RUN_ID, "steps_completed": latest_step(), "max_steps": MAX_STEPS,
    "extra_args": EXTRA_ARGS, "manifest": manifest,
    "eval": ev.to_dict(orient="records") if len(ev) else [],
    "checks": checks,
    "final_metrics_row": m.iloc[-1].to_dict() if len(m) else None,
}
with open(f"{RESULTS}/summary.json", "w") as f:
    json.dump(summary, f, indent=2, default=str)
if len(m):
    m.to_csv(f"{RESULTS}/metrics.csv", index=False)
print(f"wrote {RESULTS}/summary.json, metrics.csv, training_curves.png  ({elapsed_h():.2f} h total)")